# UKB TREX1-lupus analyses on UKB RAP

# Load libraries

In [ ]:
install.packages("logistf")
install.packages("forestploter")

In [ ]:
library(logistf)
library(forestploter)
library(dplyr)
library(car)

# Load data sets

In [ ]:
# Phenotypes and genotypes for smaller set of variants
system('dx download df_phenotypes_small.csv')
df_phenotypes_small = read.csv("df_phenotypes_small.csv")

In [ ]:
# Phenotypes and genotypes for larger set of variants
system('dx download df_phenotypes_trex1.csv')
df_phenotypes_trex1 = read.csv("df_phenotypes_trex1.csv")

In [ ]:
# Proteomics
system('dx download olink_instance_0_3072full.csv')
olink_instance_0 = read.csv("olink_instance_0_3072full.csv")

# Analyses on smaller set of variants

## Descriptive analysis

In [ ]:
# Age
df_phenotypes_small %>%
  group_by(carrier_trex1) %>%
  summarize(
    mean_age = mean(age_inception, na.rm = TRUE),
    sd_age = sd(age_inception, na.rm = TRUE),
    min_age = min(age_inception, na.rm = TRUE),
    max_age = max(age_inception, na.rm = TRUE)
  )

In [ ]:
# Sex 
df_phenotypes_small %>%
  group_by(carrier_trex1) %>%
  summarize(
    count_f = sum(rep_sex == "Female"),
    perc_f = count_f / n() * 100
  )

In [ ]:
# Ethnic background
df_phenotypes_small %>%
  group_by(carrier_trex1) %>%
  summarize(
    count_asian = sum(ethnic_1digit == "Asian or Asian British"),
    percent_asian = count_asian / n() * 100,
    count_black = sum(ethnic_1digit == "Black or Black British"),
    percent_black = count_black / n() * 100,
    count_chinese = sum(ethnic_1digit == "Chinese"),
    percent_chinese = count_chinese / n() * 100,
    count_mixed = sum(ethnic_1digit == "Mixed"),
    percent_mixed = count_mixed / n() * 100,
    count_other = sum(ethnic_1digit == "Other ethnic group"),
    percent_other = count_other / n() * 100,
    count_white = sum(ethnic_1digit == "White"),
    percent_white = count_white / n() * 100,
    count_missing = sum(ethnic_1digit == ''),
    percent_missing = count_missing / n() * 100
  )

In [ ]:
# Count outcomes
df_phenotypes_small %>%
  group_by(carrier_trex1) %>%
  summarize(
    across(ends_with("_status"), 
           list(count_1 = ~ sum(. == 1, na.rm = TRUE),
                percent_1 = ~ sum(. == 1, na.rm = TRUE) / n() * 100,
                count_0 = ~ sum(. == 0, na.rm = TRUE),
                percent_0 = ~ sum(. == 0, na.rm = TRUE) / n() * 100))
  )

## Logistic regressions

In [ ]:
# Complete case analysis
df_regression_small <- df_phenotypes_small[complete.cases(df_phenotypes_small[, 
                    c("pca1", "pca2", "pca3", "pca4", "pca5", "pca6", "pca7",
                      "pca8", "pca9", "pca10", "age_inception", "rep_sex_bin")]), ]

In [ ]:
# Loop through outcomes to generate results
# 'man': manually curated
status_columns <- c("man_sle_status", "man_slesjo_status", "man_slesjo_noselfreport_status", "man_sjo_status")

result_df <- data.frame()

for (col in status_columns) {
  # Define the formula
  formula <- as.formula(paste(col, "~ carrier_trex1 + age_inception + rep_sex_bin + pca1 + pca2 + pca3 + pca4 + pca5 + pca6 + pca7 + pca8 + pca9 + pca10"))

  # Fit the logistic regression model
  lr_result <- logistf(formula = formula,
                       data = df_regression_small,
                       pl = TRUE,
                       alpha = 0.05,
                       firth = TRUE)

  # Extract the required values and create a data frame
  result_row <- data.frame(Phenotype = col,
                           OR = exp(lr_result$coefficients['carrier_trex1']),
                           LB = exp(lr_result$ci.lower['carrier_trex1']),
                           UB = exp(lr_result$ci.upper['carrier_trex1']),
                           p = lr_result$prob['carrier_trex1'])

  # Append the result for the current status column to the final result data frame
  result_df <- rbind(result_df, result_row)
}

# Analyses on larger set of variants

## Descriptive analysis

In [ ]:
# Age
df_phenotypes_trex1 %>%
  group_by(carrier) %>%
  summarize(
    mean_age = mean(age_inception, na.rm = TRUE),
    sd_age = sd(age_inception, na.rm = TRUE),
    min_age = min(age_inception, na.rm = TRUE),
    max_age = max(age_inception, na.rm = TRUE)
  )

In [ ]:
# Sex 
df_phenotypes_trex1 %>%
  group_by(carrier) %>%
  summarize(
    count_f = sum(rep_sex == "Female"),
    perc_f = count_f / n() * 100
  )

In [ ]:
# Ethnic background
df_phenotypes_trex1 %>%
  group_by(carrier) %>%
  summarize(
    count_asian = sum(ethnic_1digit == "Asian or Asian British"),
    percent_asian = count_asian / n() * 100,
    count_black = sum(ethnic_1digit == "Black or Black British"),
    percent_black = count_black / n() * 100,
    count_chinese = sum(ethnic_1digit == "Chinese"),
    percent_chinese = count_chinese / n() * 100,
    count_mixed = sum(ethnic_1digit == "Mixed"),
    percent_mixed = count_mixed / n() * 100,
    count_other = sum(ethnic_1digit == "Other ethnic group"),
    percent_other = count_other / n() * 100,
    count_white = sum(ethnic_1digit == "White"),
    percent_white = count_white / n() * 100,
    count_missing = sum(ethnic_1digit == ''),
    percent_missing = count_missing / n() * 100
  )

In [ ]:
# Count outcomes
df_phenotypes_trex1 %>%
  group_by(carrier) %>%
  summarize(
    across(ends_with("_status"), 
           list(count_1 = ~ sum(. == 1, na.rm = TRUE),
                percent_1 = ~ sum(. == 1, na.rm = TRUE) / n() * 100,
                count_0 = ~ sum(. == 0, na.rm = TRUE),
                percent_0 = ~ sum(. == 0, na.rm = TRUE) / n() * 100))
  )

## MIRO score

In [ ]:
# Define 3 ISGs
isg_list <- c('ddx58', 'ifit3', 'siglec1')
isg_list_std <- c('ddx58_std', 'ifit3_std', 'siglec1_std')

In [ ]:
# Merge dfs
df_isg <- olink_instance_0 %>% select(all_of(isg_list))
df_phenotypes_trex1 <- left_join(df_phenotypes_trex1, df_isg, by = "eid")

In [ ]:
# Standardize
for (col_name in isg_list) {
  df_phenotypes_trex1[[paste0(col_name, "_std")]] <- as.numeric(scale(df_phenotypes_trex1[[col_name]]))
}

In [ ]:
# Check distribution
create_plot <- function(isg) {
  plot(df_phenotypes_trex1[[isg]], df_phenotypes_trex1[[paste0(isg, "_std")]],
       xlab = isg, ylab = paste0(isg, "_std"),
       main = paste("Scatter plot of", isg, "and its standardized version"))
}
lapply(isg_list, create_plot)

In [ ]:
# Subset for complete cases across selected ISGs
df_phenotypes_trex1_miro <- df_phenotypes_trex1[complete.cases(df_phenotypes_trex1[, isg_list_std]), ]

In [ ]:
# Create score
df_phenotypes_trex1_miro <- df_phenotypes_trex1_miro %>%
  mutate(score_std = (siglec1_std * 0.287) + (ddx58_std * 0.254) + (ifit3_std * 0.097),
         score_std_av = scale(score_std, center = TRUE, scale = TRUE))

## MIRO by disease

In [ ]:
# Density plots - by genotype
ggplot(df_phenotypes_trex1_miro, aes(x = score_std_av, fill = carrier)) +
    geom_density(alpha = 0.4) +
    labs(x = "MIRO score", y = "Density", title = "") +
    theme_classic() +
    scale_fill_manual(values = c("blue", "red")) +
    theme(legend.position = "none")

In [ ]:
# Linear regression
formula_right = "carrier + age_inception + rep_sex_bin + pca1 + pca2 + pca3 + pca4 + pca5 + pca6 + pca7 + pca8 + pca9 + pca10"
lm_isg <- lm(formula = as.formula(paste("score_std_av", "~", formula_right)), data = df_phenotypes_trex1_miro)
summary(lm_isg)
vif(lm_isg)
plot(fitted(lm_isg), resid(lm_isg), xlab="Fitted values", ylab="Residuals", pch = 19)
qqnorm(resid(lm_isg), pch = 19)

In [ ]:
# Density plots - by SLE/SS status
ggplot(df_phenotypes_trex1_miro, aes(x = score_std_av, fill = man_slesjo_status)) +
    geom_density(alpha = 0.4) +
    labs(x = "MIRO score", y = "Density", title = "") +
    theme_classic() +
    scale_fill_manual(values = c("blue", "red")) +
    theme(legend.position = "none")

In [ ]:
# Linear regression
formula_right = "man_slesjo_status + age_inception + rep_sex_bin"
lm_slesjo <- lm(formula = as.formula(paste("score_std_av", "~", formula_right)), data = df_phenotypes_trex1_miro)
summary(lm_slesjo)
vif(lm_slesjo)
plot(fitted(lm_slesjo), resid(lm_slesjo), xlab="Fitted values", ylab="Residuals", pch = 19)
qqnorm(resid(lm_slesjo), pch = 19)

## Logistic regressions

In [ ]:
# Complete case analysis
df_phenotypes_trex1 <- df_phenotypes_trex1[complete.cases(df_phenotypes_trex1[, 
                    c("pca1", "pca2", "pca3", "pca4", "pca5", "pca6", "pca7",
                      "pca8", "pca9", "pca10", "age_inception", "rep_sex_bin")]), ]

In [ ]:
status_columns <- c("any_neurosle_status", 'any_high_status', 'any_moderate_status', 'any_weaker_status', 'Sarcoidosis_status',
    "epi_status", "migraine_status", "depression_status", "psychosis_status", "is_status", "vascd_status", 
    "man_polymyalgiarheumatica_status", "man_ankylosingspondylitis_status", "vitiligo_status", 
    "gra_status", "psoriasis_status", "multiplesclerosis_status", "thy_status", "vasculitis_status", "man_perniciousanaemia_status", 
    "ibd_status", "myasthgravis_status", "man_pbc_status", "derm_status", "man_sle_status", "man_sjo_status", "sscl_status", 
    "man_addison_status", "t1db_status", "man_coeliac_status", "rheumart_status")

result_df <- data.frame()

for (col in status_columns) {
  # Define the formula
  formula <- as.formula(paste(col, "~ carrier + age_inception + rep_sex_bin + pca1 + pca2 + pca3 + pca4 + pca5 + pca6 + pca7 + pca8 + pca9 + pca10"))

  # Fit the logistic regression model
  lr_result <- logistf(formula = formula,
                       data = df_phenotypes_trex1,
                       pl = TRUE,
                       alpha = 0.05,
                       firth = TRUE)

  # Extract the required values and create a data frame
  result_row <- data.frame(Phenotype = col,
                           OR = exp(lr_result$coefficients['carrier']),
                           LB = exp(lr_result$ci.lower['carrier']),
                           UB = exp(lr_result$ci.upper['carrier']),
                           p = lr_result$prob['carrier'])

  # Append the result for the current status column to the final result data frame
  result_df <- rbind(result_df, result_row)
}

In [ ]:
# Forest plot
# Counts non-carriers
df_results_noncarriers <- df_phenotypes_trex1 %>%
  filter(carrier == 0) %>%
  select(all_of(status_columns))

df_results_noncarriers_counts <- lapply(df_results_noncarriers, function(col) {
  n <- sum(col == 1)
  perc <- round((n / nrow(df_results_noncarriers)) * 100, 1)
  return(data.frame(n = n, perc = perc))
}) %>%
  bind_rows(.id = "pheno_code")

df_results_noncarriers_counts$perc <- sprintf("%.1f", df_results_noncarriers_counts$perc)
df_results_noncarriers_counts$perc <- as.character(gsub(" ", "", df_results_noncarriers_counts$perc))
df_results_noncarriers_counts$n <- as.character(format(df_results_noncarriers_counts$n, big.mark = ","))
df_results_noncarriers_counts$n <- gsub(" ", "", df_results_noncarriers_counts$n)

df_results_noncarriers_counts <- df_results_noncarriers_counts %>%
  mutate(`Non-carriers, n (%)` = paste0(n, 
                                   " (", perc, ")"))

# Counts carriers
df_results_carriers <- df_phenotypes_trex1 %>%
  filter(carrier == 1) %>%
  select(all_of(status_columns))

df_results_carriers_counts <- lapply(df_results_carriers, function(col) {
  n <- sum(col == 1)
  perc <- round((n / nrow(df_results_carriers)) * 100, 1)
  return(data.frame(n = n, perc = perc))
}) %>%
  bind_rows(.id = "pheno_code")

# Remove spaces
df_results_carriers_counts$perc <- sprintf("%.1f", df_results_carriers_counts$perc)
df_results_carriers_counts$perc <- as.character(gsub(" ", "", df_results_carriers_counts$perc))
df_results_carriers_counts$n <- as.character(format(df_results_carriers_counts$n, big.mark = ","))
df_results_carriers_counts$n <- gsub(" ", "", df_results_carriers_counts$n)

# New column
df_results_carriers_counts <- df_results_carriers_counts %>%
  mutate(`Carriers, n (%)` = paste0(n, 
                                   " (", perc, ")"))


df_forestplot_trex1_large <- left_join(result_df, df_results_carriers_counts %>% 
                                select(Phenotype, `Carriers, n (%)`), by='Phenotype')

df_forestplot_trex1_large <- left_join(df_forestplot_trex1_large, df_results_noncarriers_counts %>% 
                                select(Phenotype, `Non-carriers, n (%)`), by='Phenotype')

tm <- forest_theme(core = list(fg_params = list(hjust = 0),
                               bg_params = list(fill ="#FFFFFF")),
                   rowhead = list(fg_params = list(hjust = 0, x = 0)))

forest(df_forestplot_trex1_large[,c(1:4, 8:9)],
  est = df_forestplot_trex1_large$OR,
  lower = df_forestplot_trex1_large$LB, 
  upper = df_forestplot_trex1_large$UB,
  ref_line = 1,
  ci_column = 4,
  xlim = c(0.05, 10),
  xlab = "OR",
  x_trans='log10',
  ticks_at = c(0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10),
  theme=tm)

## Linear regressions

In [ ]:
# HipV
formula_hipv = "hipv_std ~ carrier + age_inception + rep_sex_bin + hipv_centre +
    pca1 + pca2 + pca3 + pca4 + pca5 + pca6 + pca7 + pca8 + pca9 + pca10"
lm_hipv <- lm(formula = formula_hipv, data = df_phenotypes_trex1)
summary(lm_hipv)
vif(lm_hipv)
plot(fitted(lm_hipv), resid(lm_hipv), xlab="Fitted values", ylab="Residuals", pch = 19)
qqnorm(resid(lm_hipv), pch = 19)

In [ ]:
# BV
formula_bv = "bv_std ~ carrier + age_inception + rep_sex_bin + bv_centre +
    pca1 + pca2 + pca3 + pca4 + pca5 + pca6 + pca7 + pca8 + pca9 + pca10"
lm_bv <- lm(formula = formula_bv, data = df_phenotypes_trex1)
summary(lm_bv)
vif(lm_bv)
plot(fitted(lm_bv), resid(lm_bv), xlab="Fitted values", ylab="Residuals", pch = 19)
qqnorm(resid(lm_bv), pch = 19)

In [ ]:
# WMHV
formula_wmhv = "wmhv_logstd ~ carrier + age_inception + rep_sex_bin + wmhv_centre +
    pca1 + pca2 + pca3 + pca4 + pca5 + pca6 + pca7 + pca8 + pca9 + pca10"
lm_wmhv <- lm(formula = formula_wmhv, data = df_phenotypes_trex1)
summary(lm_wmhv)
vif(lm_wmhv)
plot(fitted(lm_wmhv), resid(lm_wmhv), xlab="Fitted values", ylab="Residuals", pch = 19)
qqnorm(resid(lm_wmhv), pch = 19)

In [ ]:
# Forest plot
result_df_radio <- data.frame(
code = c("hipv", "bv", "wmhv"),
Phenotype = c("Hippocampal grey matter volume (mL),\n mean (SD)",
             "Total brain volume (mL),\n mean (SD)",
              "White matter hyperintensity volume (mL),\n median (IQR)"),
`Non-carriers`= NA,
`Carriers`= NA,
` ` = paste(rep(" ", 30), collapse = " "),
beta = NA,
lb = NA,
ub = NA,
`Beta coefficient (95% CI)`= NA,
`p-value`= NA)

colnames(result_df_radio) <- c("Code", "Phenotype", "Non-carriers", "Carriers", " ", "beta", "LB", "UB", "Beta (95% CI)", "p-value")

# Add coefficients
result_df_radio$beta <- c(lm_hipv$coefficients['carrier'], lm_bv$coefficients['carrier'], lm_wmhv$coefficients['carrier'])
result_df_radio$LB <- c(confint(lm_hipv)[2,1], confint(lm_bv)[2,1], confint(lm_wmhv)[2,1])
result_df_radio$UB <- c(confint(lm_hipv)[2,2], confint(lm_bv)[2,2], confint(lm_wmhv)[2,2])

vector_hipv <- paste(format(as.character(round(lm_hipv$coefficients['carrier'],2)), nsmall=2),
                    " (", format(as.character(round(confint(lm_hipv)[2,1],2)), nsmall=2), ", ",
                    format(as.character(round(confint(lm_hipv)[2,2],2)), nsmall=2), ")", sep = "")
vector_bv <- paste(format(as.character(round(lm_bv$coefficients['carrier'],2)), nsmall=2),
                    " (", format(as.character(round(confint(lm_bv)[2,1],2)), nsmall=2), ", ",
                    format(as.character(round(confint(lm_bv)[2,2],2)), nsmall=2), ")", sep = "")
vector_wmhv <- paste(format(as.character(round(lm_wmhv$coefficients['carrier'],2)), nsmall=2),
                    " (", format(as.character(round(confint(lm_wmhv)[2,1],2)), nsmall=2), ", ",
                    format(as.character(round(confint(lm_wmhv)[2,2],2)), nsmall=2), ")", sep = "")

result_df_radio[,9] <- c(vector_hipv, vector_bv, vector_wmhv)

# Add p-values
sum_hipv <- summary(lm_hipv)
p_hipv <- round(sum_hipv$coefficients['carrier', "Pr(>|t|)"],3)
sum_bv <- summary(lm_bv)
p_bv <- round(sum_bv$coefficients['carrier', "Pr(>|t|)"],3)
sum_wmhv <- summary(lm_wmhv)
p_wmhv <- round(sum_wmhv$coefficients['carrier', "Pr(>|t|)"],3)

result_df_radio[,10] <- c(p_hipv, p_bv, p_wmhv)

# Add distribution
summary_radio <- df_phenotypes_trex1 %>% 
  mutate(bv = bv/1000, # in mL
        hipv = hipv/1000,
        wmhv = wmhv/1000) %>%
  group_by(carrier) %>%
  summarize(
    median_wmhv = median(wmhv, na.rm = TRUE),
    Q1_wmhv = quantile(wmhv, probs = 0.25, na.rm = TRUE),
    Q3_wmhv = quantile(wmhv, probs = 0.75, na.rm = TRUE),
    mean_bv = mean(bv, na.rm = TRUE),
    sd_bv = sd(bv, na.rm = TRUE),
    mean_hipv = mean(hipv, na.rm = TRUE),
    sd_hipv = sd(hipv, na.rm = TRUE),
  )

result_df_radio[,3] <- c(
paste(as.character(round(summary_radio[1, 'mean_hipv'],2)), " (", 
      as.character(round(summary_radio[1, 'sd_hipv'],2)), ")", sep=""),
paste(format(round(pull(summary_radio[1, 'mean_bv']),0), big.mark = ","), " (",
     as.character(round(summary_radio[1, 'sd_bv'],0)), ")", sep=""),
paste(as.character(round(summary_radio[1, 'median_wmhv'],2)), " (", 
     as.character(round(pull(summary_radio[1, 'Q1_wmhv']),2)), "-",  
      as.character(round(pull(summary_radio[1, 'Q3_wmhv']),2)), ")", sep=""))

result_df_radio[,4] <- c(
paste(as.character(round(summary_radio[2, 'mean_hipv'],2)), " (", 
      as.character(format(round(pull(summary_radio[2, 'sd_hipv']),2), nsmall=2)), ")", sep=""),
paste(format(round(pull(summary_radio[2, 'mean_bv']),0), big.mark = ","), " (",
     as.character(round(summary_radio[2, 'sd_bv'],0)), ")", sep=""),
paste(as.character(round(summary_radio[2, 'median_wmhv'],2)), " (", 
     as.character(round(pull(summary_radio[2, 'Q1_wmhv']),2)), "-",  
      as.character(round(pull(summary_radio[2, 'Q3_wmhv']),2)), ")", sep=""))

forest(result_df_radio,
  est = as.numeric(result_df_radio$beta),
  lower = as.numeric(result_df_radio$LB), 
  upper = as.numeric(result_df_radio$UB),
  ref_line = 0,
  ci_column = 4,
  xlim = c(-0.35, 0.35),
  ticks_at = c(-0.3, -0.15, 0, 0.15, 0.3),
  xlab = "Beta",
  theme=tm)